# 03a Setup Fixation


In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)


In [ ]:
setup_path = OUTPUT_DIR / 'metadata/setup_decisions.json'
setup = validate_setup_decisions(setup_path)

splits_dir = DATA_DIR / 'phase_2_splits'
output_data = DATA_DIR / 'phase_3_experiments'
output_data.mkdir(parents=True, exist_ok=True)

for city, split in [
    ('berlin', 'train'),
    ('berlin', 'val'),
    ('berlin', 'test'),
    ('leipzig', 'finetune'),
    ('leipzig', 'test'),
]:
    df, meta = ablation.prepare_ablation_dataset(
        base_path=splits_dir,
        city=city,
        split=split,
        setup_decisions=setup,
        return_metadata=True,
    )
    df.to_parquet(output_data / f"{city}_{split}.parquet")

log_path = OUTPUT_DIR / 'logs/03a_setup_fixation.json'
log_path.write_text(json.dumps({'status': 'completed', 'setup_decisions': setup}, indent=2))
